# Problem Framing: Hypertension Triage Screening Model (US-10)

## 1. Objective & Clinical Context
Ngozi Eze has requested a predictive triage model to prioritize individuals for blood pressure screening. The goal is to estimate whether a patient has high blood pressure using non-invasive, low-friction information that a community nurse can gather within two minutes before an actual cuff measurement is taken.

---

## 2. Target Variable Definition
* **Target Label:** `HIGH_BP_FLAG` (Binary Classification: `1` = High BP / Hypertensive, `0` = Normotensive).
* **Clinical Criteria:** Defined according to standard clinical thresholds:
  $$\text{HIGH\_BP\_FLAG} = 1 \quad \text{if} \quad (\text{SYSTOLIC\_BP} \ge 140 \lor \text{DIASTOLIC\_BP} \ge 90), \quad \text{else} \quad 0$$

---

## 3. Permitted Features (Nurse Triage Features)
These inputs can be obtained in under two minutes via quick interview or basic measurements without measuring blood pressure:
* `AGE`: Age of the participant in years.
* `NIGERIAN_STATE` / `GEOPOLITICAL_ZONE`: Regional geography capturing local demographic factors.
* `HOUSEHOLD_INCOME`: Socioeconomic status tier.
* `BMI` / `WEIGHT_KG`: Body Mass Index and weight measurements taken rapidly via a scale and stadiometer.
* `SMOKING_STATUS`: Self-reported tobacco usage habit.
* `ALCOHOL_CONSUMPTION`: Self-reported alcohol frequency (if present in dataset).
* `PHYSICAL_ACTIVITY`: Self-reported physical activity levels (if present in dataset).
* `FAMILY_HISTORY`: Self-reported history of cardiovascular disease or hypertension (if present in dataset).

---

## 4. Explicit Data Leakage Exclusions
Data leakage occurs when features containing direct knowledge of the target (which would not be available prior to screening in real life) are included during training. The following features **must be strictly excluded** from the feature matrix:

1. **`SYSTOLIC_BP`**: Direct measurement of the blood pressure, which mathematically defines the target label and would trivialise the prediction artificially.
2. **`DIASTOLIC_BP`**: Direct component of the blood pressure measurement, which directly determines the positive class boundary.
3. **`DIAGNOSED_HYPERTENSION`**: Reflects clinical knowledge or past medical diagnoses of high blood pressure, introducing direct post-diagnosis label leakage.
4. **`BP_CATEGORY`**: Derived categorization (e.g., Normal, Stage 1, Stage 2) created directly from systolic and diastolic readings.
5. **`SAMPLE_ID` / Identifier columns**: Unique administrative record IDs that carry no generalizable clinical signal and risk overfitting on sequence patterns.

In [2]:
import pandas as pd
import numpy as np

# 1. Load the summary sample dataset
df = pd.read_csv(r"C:\Users\ugochukwu.emenike\PycharmProjects\PythonProject\powerbi\analysis_sample.csv")  # or specify your path

# 2. Construct the binary target
df['HIGH_BP_FLAG'] = np.where(
    (df['SYSTOLIC_BP'] >= 140) | (df['DIASTOLIC_BP'] >= 90), 1, 0
)

# 3. Explicitly define leakage columns to drop
leakage_cols = [
    'SYSTOLIC_BP',
    'DIASTOLIC_BP',
    'DIAGNOSED_HYPERTENSION',
    'BP_CATEGORY',
    'SAMPLE_ID'
]

# 4. Separate features (X) and target (y)
X = df.drop(columns=leakage_cols + ['HIGH_BP_FLAG'])
y = df['HIGH_BP_FLAG']

print("Features selected (X):", list(X.columns))
print("Target distribution (y):")
print(y.value_counts(normalize=True))

Features selected (X): ['AGE', 'NIGERIAN_STATE', 'GEOPOLITICAL_ZONE', 'HOUSEHOLD_INCOME', 'WEIGHT_KG', 'BMI', 'SMOKING_STATUS', 'MALARIA_REPORTED']
Target distribution (y):
HIGH_BP_FLAG
1    0.580161
0    0.419839
Name: proportion, dtype: float64
